<a href="https://colab.research.google.com/github/dhivarukha/amateurporto/blob/My-Sales/Cashier_Printer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install reportlab

In [3]:
import os
from datetime import datetime
import pytz
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm
import tkinter as tk
from tkinter import ttk, messagebox

In [4]:
# ==========================
# 1. Data menu
# ==========================
menu_items = {
    "Arure Matcha Latte": 23000,
    "Matcha Cold Whisk": 25000,
    "Kopi Arure": 18000,
    "Add Oatside Milk": 3000,
    "Add Vanilla Syrup": 3000
}

In [5]:
# ==========================
# 2. Ambil & update nomor order
# ==========================
def get_next_order_no():
    file = "order_counter.txt"
    today = datetime.now(pytz.timezone("Asia/Jakarta")).strftime("%Y-%m-%d")

    if not os.path.exists(file):
        with open(file, "w") as f:
            f.write(f"{today},1")
            return 1

    with open(file, "r+") as f:
        data = f.read().strip()
        if "," in data:
            last_date, last_no = data.split(",")
            last_no = int(last_no)
        else:
            # Format lama: cuma angka
            last_date, last_no = today, int(data) if data.isdigit() else 0

        if last_date == today:
            new_no = last_no + 1
        else:
            new_no = 1  # reset kalau sudah ganti hari

        f.seek(0)
        f.write(f"{today},{new_no}")
        f.truncate()
        return new_no

In [6]:
# ==========================
# 3. Cetak PDF Nota
# ==========================
def print_receipt(order_no, items, bayar):
    width = 58 * mm   # lebar fix 58mm

    # Hitung tinggi dinamis
    base_height = 100 * mm   # header + footer minimum
    item_height = (len(items) * 10) * mm/3  # tinggi tambahan per item (≈3.3 mm per item)
    height = base_height + item_height

    filename = f"struk_{order_no}.pdf"
    c = canvas.Canvas(filename, pagesize=(width, height))
    c.setFont("Helvetica", 8)

    # Header
    store_name = "Arure! Home Cafe"
    c.setFont("Helvetica-Bold", 10)
    c.drawCentredString(width/2, height-20, store_name)
    c.setFont("Helvetica", 7)
    c.drawCentredString(width/2, height-30, "Jl. Danau Maninjau Blok A No. 72")
    c.drawCentredString(width/2, height-40, "Bendungan Hilir, Jakarta Pusat")

    c.setFont("Helvetica-Bold", 12)
    c.drawCentredString(width/2, height-60, f"Bill #{order_no}")
    c.setFont("Helvetica", 8)
    c.drawString(5, height-75, "Take Away Order")

    timezone = pytz.timezone("Asia/Jakarta")
    now = datetime.now(timezone)
    tanggal = now.strftime("%A, %d %B %Y, %H:%M:%S")
    c.drawString(5, height-85, tanggal)

    # Table header
    y = height-100
    c.line(5, y, width-5, y)
    y -= 10
    c.drawString(5, y, "Item")
    c.drawRightString(width/2+10, y, "Qty")
    c.drawRightString(width-5, y, "Harga")
    y -= 5
    c.line(5, y, width-5, y)

    # Items
    total = 0
    for item in items:
        y -= 12
        # Nama item
        c.setFont("Helvetica", 8)
        c.drawString(5, y, item['name'][:25])

        # Qty
        c.setFont("Helvetica", 8)
        c.drawRightString(width/2+10, y, str(item['qty']))

        # Subtotal (harga per item)
        subtotal_item = item['qty'] * item['price']
        c.setFont("Helvetica-Bold", 8)
        c.drawRightString(width-5, y, f"Rp {subtotal_item:,}".replace(",", "."))
        total += subtotal_item


    # Summary
    y -= 10
    c.line(5, y, width-5, y)
    y -= 12
    c.setFont("Helvetica-Bold", 8)
    c.drawString(5, y, "TOTAL")
    c.drawRightString(width-5, y, f"Rp {total:,}".replace(",", "."))

    y -= 12
    c.setFont("Helvetica", 8)
    c.drawString(5, y, "TUNAI")
    c.drawRightString(width-5, y, f"Rp {bayar:,}".replace(",", "."))

    kembalian = bayar - total
    y -= 12
    c.drawString(5, y, "KEMBALIAN")
    c.drawRightString(width-5, y, f"Rp {kembalian:,}".replace(",", "."))

    # Footer
    y -= 30
    c.setFont("Helvetica", 8)
    c.drawCentredString(width/2, y, "Follow us at Tiktok & Instagram")
    y -= 10
    c.setFont("Helvetica", 8)
    c.drawCentredString(width/2, y, "arure.jkt")

    y -= 15
    c.setFont("Helvetica", 7)
    c.drawCentredString(width/2, y, "Terima Kasih")
    y -= 10
    c.setFont("Helvetica", 7)
    c.drawCentredString(width/2, y, "Ditunggu Kedatangan Berikutnya")

    c.showPage()
    c.save()
    return filename

In [7]:
# ==========================
# 4. Input Pesanan di Colab
# ==========================
items = []

print("=== Menu Arure Home Cafe ===")
for i, (name, price) in enumerate(menu_items.items(), 1):
    print(f"{i}. {name} - Rp {price:,}".replace(",", "."))

while True:
    pilihan = input("Pilih menu (nomor, enter untuk selesai): ")
    if not pilihan.strip():
        break
    qty = int(input("Jumlah: "))
    menu_name = list(menu_items.keys())[int(pilihan)-1]
    items.append({"name": menu_name, "qty": qty, "price": menu_items[menu_name]})

# Hitung total
total = sum(item["qty"] * item["price"] for item in items)

# Input uang tunai
print(f"\nTotal belanja: Rp {total:,}".replace(",", "."))
bayar = int(input("Masukkan uang tunai: "))

# Cetak struk
order_no = get_next_order_no()
filename = print_receipt(order_no, items, bayar)
print(f"Nota tersimpan: {filename}")

=== Menu Arure Home Cafe ===
1. Arure Matcha Latte - Rp 23.000
2. Matcha Cold Whisk - Rp 25.000
3. Kopi Arure - Rp 18.000
4. Add Oatside Milk - Rp 3.000
5. Add Vanilla Syrup - Rp 3.000
Pilih menu (nomor, enter untuk selesai): 2
Jumlah: 3
Pilih menu (nomor, enter untuk selesai): 4
Jumlah: 3
Pilih menu (nomor, enter untuk selesai): 

Total belanja: Rp 84.000
Masukkan uang tunai: 100000
Nota tersimpan: struk_1.pdf
